# Week 10 — From MLOps to LLMOps: Fine-Tuning Gemini on the IRIS Pipeline

This notebook implements the Week 10 LLMOps assignment.

## Objectives

1. Prepare two representations of the IRIS dataset:
   - V1 — Raw feature representation
   - V2 — Natural-language description
2. Store the fine-tuning datasets in Google Cloud Storage.
3. Fine-tune the same Gemini model using both representations.
4. Evaluate both tuned models on the same held-out test set.
5. Compare:
   - Accuracy
   - Per-class precision
   - Per-class recall
   - Format compliance
6. Analyze how data representation affects LLM performance.

## GCP Configuration

Project:
`project-0a6400db-0297-4323-a8f`

Bucket:
`gs://mlops-course-project-0a6400db-0297-4323-a8f`

Region:
`us-central1`

In [1]:
!pip install -q --upgrade google-genai google-cloud-storage scikit-learn pandas

## Imports

In [4]:
import os
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)

from google import genai
from google.genai import types
from google.cloud import storage

print("Libraries imported successfully.")

Libraries imported successfully.


## GCP configuration

In [5]:
PROJECT_ID = "project-0a6400db-0297-4323-a8f"

BUCKET_NAME = "mlops-course-project-0a6400db-0297-4323-a8f"

REGION = "us-central1"

BASE_MODEL = "gemini-2.5-flash-lite"

GCS_PREFIX = "week10"

print("Project ID :", PROJECT_ID)
print("Bucket     :", BUCKET_NAME)
print("Region     :", REGION)
print("Base model :", BASE_MODEL)

Project ID : project-0a6400db-0297-4323-a8f
Bucket     : mlops-course-project-0a6400db-0297-4323-a8f
Region     : us-central1
Base model : gemini-2.5-flash-lite


## Initialize Vertex AI client

In [6]:
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = REGION
os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "True"

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=REGION,
)

print("Vertex AI Gemini client initialized successfully.")

Vertex AI Gemini client initialized successfully.


### Verify authentication

In [7]:
import google.auth

credentials, authenticated_project = google.auth.default()

print("Authenticated project:", authenticated_project)
print("Credential type:", type(credentials).__name__)
print(
    "Credential identity:",
    getattr(
        credentials,
        "service_account_email",
        "User/ADC credential"
    )
)

Authenticated project: project-0a6400db-0297-4323-a8f
Credential type: Credentials
Credential identity: User/ADC credential


## Load Iris dataset

In [8]:
iris = load_iris()

X = iris.data
y = iris.target

feature_names = [
    "sepal_length",
    "sepal_width",
    "petal_length",
    "petal_width",
]

class_names = [
    "setosa",
    "versicolor",
    "virginica",
]

print("Number of samples:", len(X))
print("Number of features:", X.shape[1])
print("Classes:", class_names)

Number of samples: 150
Number of features: 4
Classes: ['setosa', 'versicolor', 'virginica']


## Create train/test split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

Training samples: 120
Test samples: 30


## Task 1 — V1 Raw Feature Representation

V1 represents the Iris measurements directly as text.

Example:

sepal_length: 5.1, sepal_width: 3.5, petal_length: 1.4, petal_width: 0.2

The target is the Iris species.

## Create v1 RAW representation

In [10]:
def create_v1_record(features, target):
    """
    V1:
    Serialize the four Iris measurements directly as text.
    """

    input_text = (
        f"sepal_length: {features[0]}, "
        f"sepal_width: {features[1]}, "
        f"petal_length: {features[2]}, "
        f"petal_width: {features[3]}"
    )

    output_text = class_names[target]

    return {
        "contents": [
            {
                "role": "user",
                "parts": [
                    {
                        "text": input_text
                    }
                ]
            },
            {
                "role": "model",
                "parts": [
                    {
                        "text": output_text
                    }
                ]
            }
        ]
    }


v1_train_records = [
    create_v1_record(features, target)
    for features, target in zip(X_train, y_train)
]

print("V1 training records:", len(v1_train_records))

print("\nExample V1:")
print(json.dumps(v1_train_records[0], indent=2))

V1 training records: 120

Example V1:
{
  "contents": [
    {
      "role": "user",
      "parts": [
        {
          "text": "sepal_length: 4.4, sepal_width: 2.9, petal_length: 1.4, petal_width: 0.2"
        }
      ]
    },
    {
      "role": "model",
      "parts": [
        {
          "text": "setosa"
        }
      ]
    }
  ]
}


## Task 2 — V2 Natural Language Representation

V2 expresses the same numerical measurements as a natural-language description.

Example:

"A flower specimen has a sepal length of 5.1 cm,
sepal width of 3.5 cm, petal length of 1.4 cm,
and petal width of 0.2 cm. Identify the iris species."

The target is expressed as:

"This is Iris setosa."

## Create V2 records

In [11]:
def create_v2_record(features, target):
    """
    V2:
    Express the same four measurements as natural language.
    """

    input_text = (
        f"A flower specimen has a sepal length of {features[0]} cm, "
        f"sepal width of {features[1]} cm, "
        f"petal length of {features[2]} cm, "
        f"and petal width of {features[3]} cm. "
        f"Identify the iris species."
    )

    output_text = (
        f"This is Iris {class_names[target]}."
    )

    return {
        "contents": [
            {
                "role": "user",
                "parts": [
                    {
                        "text": input_text
                    }
                ]
            },
            {
                "role": "model",
                "parts": [
                    {
                        "text": output_text
                    }
                ]
            }
        ]
    }


v2_train_records = [
    create_v2_record(features, target)
    for features, target in zip(X_train, y_train)
]

print("V2 training records:", len(v2_train_records))

print("\nExample V2:")
print(json.dumps(v2_train_records[0], indent=2))

V2 training records: 120

Example V2:
{
  "contents": [
    {
      "role": "user",
      "parts": [
        {
          "text": "A flower specimen has a sepal length of 4.4 cm, sepal width of 2.9 cm, petal length of 1.4 cm, and petal width of 0.2 cm. Identify the iris species."
        }
      ]
    },
    {
      "role": "model",
      "parts": [
        {
          "text": "This is Iris setosa."
        }
      ]
    }
  ]
}


## Validate V1/V2

In [12]:
def validate_record(record):

    assert "contents" in record

    contents = record["contents"]

    assert len(contents) == 2

    assert contents[0]["role"] == "user"
    assert contents[1]["role"] == "model"

    assert "parts" in contents[0]
    assert "parts" in contents[1]

    assert "text" in contents[0]["parts"][0]
    assert "text" in contents[1]["parts"][0]


for record in v1_train_records:
    validate_record(record)

for record in v2_train_records:
    validate_record(record)

print("V1: VALID")
print("V2: VALID")
print("Gemini tuning schema validation passed.")

V1: VALID
V2: VALID
Gemini tuning schema validation passed.


### Create data directory

In [13]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print("Data directory:", DATA_DIR.resolve())

Data directory: /home/jupyter/23F1001572_MLOPS_WEEKLY_ASSIGNMENT/data


### Write JSONL files

In [14]:
v1_train_path = DATA_DIR / "iris_v1_train.jsonl"
v2_train_path = DATA_DIR / "iris_v2_train.jsonl"


def write_jsonl(records, path):

    with open(path, "w") as f:
        for record in records:
            f.write(json.dumps(record) + "\n")


write_jsonl(
    v1_train_records,
    v1_train_path
)

write_jsonl(
    v2_train_records,
    v2_train_path
)

print("Saved:")
print(v1_train_path)
print(v2_train_path)

Saved:
data/iris_v1_train.jsonl
data/iris_v2_train.jsonl


## Verify files

In [15]:
for path in [
    v1_train_path,
    v2_train_path
]:

    with open(path) as f:
        lines = f.readlines()

    print(
        path.name,
        "->",
        len(lines),
        "records"
    )

iris_v1_train.jsonl -> 120 records
iris_v2_train.jsonl -> 120 records


## Upload to YOUR GCS bucket

In [18]:
storage_client = storage.Client(
    project=PROJECT_ID
)

bucket = storage_client.bucket(
    BUCKET_NAME
)


def upload_file(local_path, blob_name):

    blob = bucket.blob(blob_name)

    blob.upload_from_filename(
        str(local_path)
    )

    return (
        f"gs://{BUCKET_NAME}/{blob_name}"
    )


v1_train_uri = upload_file(
    v1_train_path,
    f"{GCS_PREFIX}/data/iris_v1_train.jsonl"
)

v2_train_uri = upload_file(
    v2_train_path,
    f"{GCS_PREFIX}/data/iris_v2_train.jsonl"
)

print("V1 URI:")
print(v1_train_uri)

print("\nV2 URI:")
print(v2_train_uri)

V1 URI:
gs://mlops-course-project-0a6400db-0297-4323-a8f/week10/data/iris_v1_train.jsonl

V2 URI:
gs://mlops-course-project-0a6400db-0297-4323-a8f/week10/data/iris_v2_train.jsonl


## Verify GCS

In [19]:
print("Objects in GCS:")
print("-" * 50)

for blob in bucket.list_blobs(
    prefix=f"{GCS_PREFIX}/data/"
):

    print(blob.name)

Objects in GCS:
--------------------------------------------------
week10/data/iris_v1_test.jsonl
week10/data/iris_v1_train.jsonl
week10/data/iris_v2_test.jsonl
week10/data/iris_v2_train.jsonl


## Fine-tuning configuration

In [20]:
EPOCH_COUNT = 2
LEARNING_RATE_MULTIPLIER = 1.0

print("Base model:", BASE_MODEL)
print("Epochs:", EPOCH_COUNT)
print(
    "Learning rate multiplier:",
    LEARNING_RATE_MULTIPLIER
)

Base model: gemini-2.5-flash-lite
Epochs: 2
Learning rate multiplier: 1.0


## Submit v1 tuning job

In [21]:
tuning_job_v1 = client.tunings.tune(
    base_model=BASE_MODEL,
    training_dataset=types.TuningDataset(
        gcs_uri=v1_train_uri
    ),
    config=types.CreateTuningJobConfig(
        tuned_model_display_name="iris-gemini-v1-raw",
        epoch_count=EPOCH_COUNT,
        learning_rate_multiplier=LEARNING_RATE_MULTIPLIER,
    ),
)

print("V1 tuning job submitted successfully.")
print("Job name:", tuning_job_v1.name)
print("State:", tuning_job_v1.state)

/var/tmp/ipykernel_16001/281276161.py:1: ExperimentalWarning: The SDK's tuning implementation is experimental, and may change in future versions.
  tuning_job_v1 = client.tunings.tune(


V1 tuning job submitted successfully.
Job name: projects/938959281806/locations/us-central1/tuningJobs/7336942282131636224
State: JobState.JOB_STATE_PENDING


## Submit V2 tuning job

In [22]:
tuning_job_v2 = client.tunings.tune(
    base_model=BASE_MODEL,
    training_dataset=types.TuningDataset(
        gcs_uri=v2_train_uri
    ),
    config=types.CreateTuningJobConfig(
        tuned_model_display_name="iris-gemini-v2-description",
        epoch_count=EPOCH_COUNT,
        learning_rate_multiplier=LEARNING_RATE_MULTIPLIER,
    ),
)

print("V2 tuning job submitted successfully.")
print("Job name:", tuning_job_v2.name)
print("State:", tuning_job_v2.state)

V2 tuning job submitted successfully.
Job name: projects/938959281806/locations/us-central1/tuningJobs/3095677333055471616
State: JobState.JOB_STATE_PENDING


In [30]:
v1_job = client.tunings.get(name=tuning_job_v1.name)
v2_job = client.tunings.get(name=tuning_job_v2.name)

print("V1:", v1_job.state)
print("V2:", v2_job.state)

V1: JobState.JOB_STATE_SUCCEEDED
V2: JobState.JOB_STATE_SUCCEEDED


In [31]:
v1_job = client.tunings.get(
    name=tuning_job_v1.name
)

print("V1")
print("-" * 40)
print("Job:", v1_job.name)
print("State:", v1_job.state)

if "tuning_job_v2" in globals():
    v2_job = client.tunings.get(
        name=tuning_job_v2.name
    )

    print("\nV2")
    print("-" * 40)
    print("Job:", v2_job.name)
    print("State:", v2_job.state)

V1
----------------------------------------
Job: projects/938959281806/locations/us-central1/tuningJobs/7336942282131636224
State: JobState.JOB_STATE_SUCCEEDED

V2
----------------------------------------
Job: projects/938959281806/locations/us-central1/tuningJobs/3095677333055471616
State: JobState.JOB_STATE_SUCCEEDED


In [32]:
v1_job = client.tunings.get(
    name=tuning_job_v1.name
)

v2_job = client.tunings.get(
    name=tuning_job_v2.name
)

print("V1")
print("-" * 50)
print("State:", v1_job.state)
print("Tuned model:", v1_job.tuned_model)

print("\nV2")
print("-" * 50)
print("State:", v2_job.state)
print("Tuned model:", v2_job.tuned_model)

V1
--------------------------------------------------
State: JobState.JOB_STATE_SUCCEEDED
Tuned model: model='projects/938959281806/locations/us-central1/models/5605838593739194368@1' endpoint='projects/938959281806/locations/us-central1/endpoints/6507952699957313536' checkpoints=[TunedModelCheckpoint(
  checkpoint_id='1',
  endpoint='projects/938959281806/locations/us-central1/endpoints/6856981671078526976',
  epoch=1,
  step=1
), TunedModelCheckpoint(
  checkpoint_id='2',
  endpoint='projects/938959281806/locations/us-central1/endpoints/6507952699957313536',
  epoch=2,
  step=2
)]

V2
--------------------------------------------------
State: JobState.JOB_STATE_SUCCEEDED
Tuned model: model='projects/938959281806/locations/us-central1/models/8320383269136760832@1' endpoint='projects/938959281806/locations/us-central1/endpoints/3166281776448405504' checkpoints=[TunedModelCheckpoint(
  checkpoint_id='1',
  endpoint='projects/938959281806/locations/us-central1/endpoints/506680081919875481

In [33]:
V1_ENDPOINT = (
    "projects/938959281806/locations/us-central1/"
    "endpoints/6507952699957313536"
)

V2_ENDPOINT = (
    "projects/938959281806/locations/us-central1/"
    "endpoints/3166281776448405504"
)

print("V1 Endpoint:")
print(V1_ENDPOINT)

print("\nV2 Endpoint:")
print(V2_ENDPOINT)

V1 Endpoint:
projects/938959281806/locations/us-central1/endpoints/6507952699957313536

V2 Endpoint:
projects/938959281806/locations/us-central1/endpoints/3166281776448405504


## Load V1/V2 test data

In [34]:
v1_test_blob = bucket.blob(
    "week10/data/iris_v1_test.jsonl"
)

v2_test_blob = bucket.blob(
    "week10/data/iris_v2_test.jsonl"
)

v1_test_lines = (
    v1_test_blob.download_as_text()
    .strip()
    .splitlines()
)

v2_test_lines = (
    v2_test_blob.download_as_text()
    .strip()
    .splitlines()
)

v1_test_records = [
    json.loads(line)
    for line in v1_test_lines
]

v2_test_records = [
    json.loads(line)
    for line in v2_test_lines
]

print("V1 test samples:", len(v1_test_records))
print("V2 test samples:", len(v2_test_records))

V1 test samples: 30
V2 test samples: 30


In [36]:
def extract_test_record(record):
    """
    Test files follow the assignment's
    input_text / output_text format.
    """

    input_text = record["input_text"]
    expected_output = record["output_text"]

    return input_text, expected_output


# V1 test data
v1_test_inputs = []
v1_expected_outputs = []

for record in v1_test_records:
    input_text, expected_output = extract_test_record(record)

    v1_test_inputs.append(input_text)
    v1_expected_outputs.append(expected_output)


# V2 test data
v2_test_inputs = []
v2_expected_outputs = []

for record in v2_test_records:
    input_text, expected_output = extract_test_record(record)

    v2_test_inputs.append(input_text)
    v2_expected_outputs.append(expected_output)


print("V1 test samples:", len(v1_test_inputs))
print("V2 test samples:", len(v2_test_inputs))

print("\nV1 example")
print("Input :", v1_test_inputs[0])
print("Target:", v1_expected_outputs[0])

print("\nV2 example")
print("Input :", v2_test_inputs[0])
print("Target:", v2_expected_outputs[0])

V1 test samples: 30
V2 test samples: 30

V1 example
Input : sepal_length: 4.4, sepal_width: 3.0, petal_length: 1.3, petal_width: 0.2
Target: setosa

V2 example
Input : A flower specimen has a sepal length of 4.4 cm, sepal width of 3.0 cm, petal length of 1.3 cm, and petal width of 0.2 cm. Identify the iris species.
Target: This is Iris setosa.


## Inference function

In [39]:
def generate_prediction(endpoint, prompt):
    response = client.models.generate_content(
        model=endpoint,
        contents=prompt,
    )
    
    return response.text.strip()

## Evaluate V1

In [ ]:
v1_outputs = []

for i, prompt in enumerate(v1_test_inputs):

    output = generate_prediction(
        V1_ENDPOINT,
        prompt
    )

    v1_outputs.append(output)

    print(
        f"{i+1:02d}. Expected: "
        f"{v1_expected_outputs[i]} | "
        f"Predicted: {output}"
    )

print("\nV1 inference completed.")

01. Expected: setosa | Predicted: This data represents the measurements of a flower. Based on these values, it is highly likely that this flower belongs to the **Iris setosa** species.

Here's why:

*   **Sepal Length (4.4 cm):** This is on the shorter side compared to other Iris species.
*   **Sepal Width (3.0 cm):** This is relatively wide.
*   **Petal Length (1.3 cm):** This is very short, a key characteristic of Iris setosa.
*   **Petal Width (0.2 cm):** This is also very narrow, another strong indicator of Iris setosa.

These measurements are very typical of the Iris setosa variety.

**If you're working with the Iris dataset, this particular observation would most likely be classified as Iris setosa.**
02. Expected: virginica | Predicted: Based on the provided sepal and petal measurements:

*   **sepal_length:** 6.1
*   **sepal_width:** 3.0
*   **petal_length:** 4.9
*   **petal_width:** 1.8

These values are characteristic of the **Iris virginica** species.

This is a common set o

## Evaluate V2

In [ ]:
v2_outputs = []

for i, prompt in enumerate(v2_test_inputs):

    output = generate_prediction(
        V2_ENDPOINT,
        prompt
    )

    v2_outputs.append(output)

    print(
        f"{i+1:02d}. Expected: "
        f"{v2_expected_outputs[i]} | "
        f"Predicted: {output}"
    )

print("\nV2 inference completed.")

01. Expected: This is Iris setosa. | Predicted: Based on the measurements provided, the flower specimen most closely matches the characteristics of **Iris setosa**.

Here's why:

*   **Sepal Length and Width:** Iris setosa typically has sepals that are relatively short and broad. Your measurements of 4.4 cm by 3.0 cm fit this description well.
*   **Petal Length and Width:** Iris setosa is known for its much shorter and narrower petals compared to its sepals. Your measurements of 1.3 cm by 0.2 cm are consistent with this.

In contrast, the other common Iris species often found in datasets like the famous Iris dataset (Iris virginica and Iris versicolor) have significantly larger petals in proportion to their sepals.

**Therefore, the iris species is most likely Iris setosa.**
02. Expected: This is Iris virginica. | Predicted: Based on the provided measurements for the flower specimen:

*   Sepal length: 6.1 cm
*   Sepal width: 3.0 cm
*   Petal length: 4.9 cm
*   Petal width: 1.8 cm

Th

## Evaluation Metrics

In [43]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)
import re


VALID_CLASSES = ["setosa", "versicolor", "virginica"]


# ------------------------------------------------------------
# Extract species from model output
# Used ONLY for semantic classification accuracy.
# ------------------------------------------------------------

def extract_species(text):

    text = text.lower()

    # Prefer the three allowed classes.
    # Ignore other species such as Iris germanica.
    for species in VALID_CLASSES:
        if re.search(rf"\b{species}\b", text):
            return species

    return "invalid"


# ------------------------------------------------------------
# V1
# ------------------------------------------------------------

v1_true = [
    extract_species(x)
    for x in v1_expected_outputs
]

v1_pred = [
    extract_species(x)
    for x in v1_outputs
]


# ------------------------------------------------------------
# V2
# ------------------------------------------------------------

v2_true = [
    extract_species(x)
    for x in v2_expected_outputs
]

v2_pred = [
    extract_species(x)
    for x in v2_outputs
]


# ------------------------------------------------------------
# Classification accuracy
# ------------------------------------------------------------

v1_accuracy = accuracy_score(v1_true, v1_pred)
v2_accuracy = accuracy_score(v2_true, v2_pred)


# ------------------------------------------------------------
# Precision / Recall
# ------------------------------------------------------------

v1_precision, v1_recall, _, _ = precision_recall_fscore_support(
    v1_true,
    v1_pred,
    labels=VALID_CLASSES,
    average=None,
    zero_division=0
)

v2_precision, v2_recall, _, _ = precision_recall_fscore_support(
    v2_true,
    v2_pred,
    labels=VALID_CLASSES,
    average=None,
    zero_division=0
)


# ------------------------------------------------------------
# STRICT FORMAT COMPLIANCE
#
# V1 expected:
#   setosa / versicolor / virginica
#
# V2 expected:
#   This is Iris setosa.
#   This is Iris versicolor.
#   This is Iris virginica.
#
# Anything else = non-compliant.
# ------------------------------------------------------------

v1_valid_formats = {
    "setosa",
    "versicolor",
    "virginica"
}

v2_valid_formats = {
    "This is Iris setosa.",
    "This is Iris versicolor.",
    "This is Iris virginica."
}


v1_format_compliant = [
    output.strip() in v1_valid_formats
    for output in v1_outputs
]

v2_format_compliant = [
    output.strip() in v2_valid_formats
    for output in v2_outputs
]


v1_format_rate = sum(v1_format_compliant) / len(v1_format_compliant)
v2_format_rate = sum(v2_format_compliant) / len(v2_format_compliant)


# ============================================================
# Display Results
# ============================================================

print("=" * 65)
print("TASK 4 — V1 vs V2 EVALUATION")
print("=" * 65)

print("\nV1 — Raw Feature Representation")
print("-" * 65)

print(f"Accuracy:          {v1_accuracy:.4f}")
print(f"Format Compliance: {v1_format_rate:.4f}")

print("\nPer-Class Metrics:")
for cls, p, r in zip(
    VALID_CLASSES,
    v1_precision,
    v1_recall
):
    print(
        f"{cls:12s} "
        f"Precision: {p:.4f} | "
        f"Recall: {r:.4f}"
    )


print("\nV2 — Natural Language Representation")
print("-" * 65)

print(f"Accuracy:          {v2_accuracy:.4f}")
print(f"Format Compliance: {v2_format_rate:.4f}")

print("\nPer-Class Metrics:")
for cls, p, r in zip(
    VALID_CLASSES,
    v2_precision,
    v2_recall
):
    print(
        f"{cls:12s} "
        f"Precision: {p:.4f} | "
        f"Recall: {r:.4f}"
    )


print("\n" + "=" * 65)
print("COMPARISON")
print("=" * 65)

print(
    f"V1 Accuracy:           {v1_accuracy:.4f}"
)

print(
    f"V2 Accuracy:           {v2_accuracy:.4f}"
)

print(
    f"V1 Format Compliance:  {v1_format_rate:.4f}"
)

print(
    f"V2 Format Compliance:  {v2_format_rate:.4f}"
)

TASK 4 — V1 vs V2 EVALUATION

V1 — Raw Feature Representation
-----------------------------------------------------------------
Accuracy:          0.5333
Format Compliance: 0.0000

Per-Class Metrics:
setosa       Precision: 0.4348 | Recall: 1.0000
versicolor   Precision: 1.0000 | Recall: 0.2000
virginica    Precision: 1.0000 | Recall: 0.4000

V2 — Natural Language Representation
-----------------------------------------------------------------
Accuracy:          0.2333
Format Compliance: 0.0000

Per-Class Metrics:
setosa       Precision: 0.7143 | Recall: 0.5000
versicolor   Precision: 0.0000 | Recall: 0.0000
virginica    Precision: 1.0000 | Recall: 0.2000

COMPARISON
V1 Accuracy:           0.5333
V2 Accuracy:           0.2333
V1 Format Compliance:  0.0000
V2 Format Compliance:  0.0000


## Final Comparison and Conclusion

In [44]:
print("=" * 70)
print("FINAL V1 vs V2 COMPARISON")
print("=" * 70)

results = {
    "V1 - Raw Features": {
        "Accuracy": v1_accuracy,
        "Format Compliance": v1_format_rate
    },
    "V2 - Natural Language": {
        "Accuracy": v2_accuracy,
        "Format Compliance": v2_format_rate
    }
}

for model_name, metrics in results.items():
    print(f"\n{model_name}")
    print(f"Accuracy:          {metrics['Accuracy']:.2%}")
    print(f"Format Compliance: {metrics['Format Compliance']:.2%}")


print("\n" + "=" * 70)
print("CONCLUSION")
print("=" * 70)

print(
    f"""
V1 achieved {v1_accuracy:.2%} accuracy, while V2 achieved
{v2_accuracy:.2%} accuracy.

Therefore, V1 (raw feature representation) performed better
than V2 (natural-language representation) on the held-out
test set.

Both models achieved {v1_format_rate:.2%} and
{v2_format_rate:.2%} format compliance respectively.

The results suggest that, for this structured Iris
classification task, directly presenting the numerical
measurements was more effective than converting the same
information into natural-language descriptions.

The zero format-compliance rate also demonstrates an
important LLM-specific failure mode: the models generated
explanations instead of consistently following the required
output format.
"""
)

FINAL V1 vs V2 COMPARISON

V1 - Raw Features
Accuracy:          53.33%
Format Compliance: 0.00%

V2 - Natural Language
Accuracy:          23.33%
Format Compliance: 0.00%

CONCLUSION

V1 achieved 53.33% accuracy, while V2 achieved
23.33% accuracy.

Therefore, V1 (raw feature representation) performed better
than V2 (natural-language representation) on the held-out
test set.

Both models achieved 0.00% and
0.00% format compliance respectively.

The results suggest that, for this structured Iris
classification task, directly presenting the numerical
measurements was more effective than converting the same
information into natural-language descriptions.

The zero format-compliance rate also demonstrates an
important LLM-specific failure mode: the models generated
explanations instead of consistently following the required
output format.

